# The Human Cost of Civilian Targeting
## 01 — Data cleaning e quality audit

**Domanda centrale:** *How has violence against civilians changed globally between 2021 and 2025, and where has humanitarian risk become most concentrated?*

Questo notebook prepara il dataset ACLED per l'analisi globale degli eventi contrassegnati come `Civilian targeting`. La pipeline è pensata per essere riproducibile e leggibile nelle slide: caricamento a blocchi, filtro, normalizzazione, controlli di qualità, feature engineering ed esportazione in Parquet.

### Note metodologiche da mantenere nel progetto

- `fatalities` indica le fatalità registrate o stimate da ACLED per l'intero evento. Non va presentata automaticamente come numero esatto di civili uccisi.
- Il tag `civilian_targeting` identifica eventi in cui i civili risultano target; non comprende ogni possibile conseguenza indiretta del conflitto sulla popolazione.
- Il file disponibile copre il 2025 soltanto fino al **25 agosto 2025**. I confronti annuali 2021–2025 devono quindi usare una finestra comune gennaio–25 agosto oppure mostrare il 2025 come anno parziale.
- Conteggio eventi e fatalità misurano fenomeni diversi: frequenza e intensità letale vanno visualizzate separatamente.

## 1. Setup

In [2]:
from pathlib import Path
import platform

import numpy as np
import pandas as pd
import pyarrow
import pycountry

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

print(f"Python: {platform.python_version()}")
print(f"pandas: {pd.__version__}")
print(f"pyarrow: {pyarrow.__version__}")

Python: 3.13.5
pandas: 2.3.2
pyarrow: 21.0.0


In [3]:
# Parametri del progetto
PROJECT_DIR = Path.cwd()
START_YEAR = 2021
END_YEAR = 2025
CHUNK_SIZE = 200_000

csv_candidates = [
    path for path in PROJECT_DIR.glob("*acled*.csv*")
    if path.is_file() and "clean" not in path.name.lower()
]
if not csv_candidates:
    raise FileNotFoundError("Nessun CSV ACLED trovato nella cartella del progetto.")

# Se ci sono più export, usa il file più grande e mostra esplicitamente la scelta.
RAW_CSV = max(csv_candidates, key=lambda path: path.stat().st_size)
OUTPUT_DIR = PROJECT_DIR / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CLEAN_PARQUET = OUTPUT_DIR / "acled_civilian_targeting_2021_2025.parquet"
AUDIT_CSV = OUTPUT_DIR / "cleaning_audit.csv"
COVERAGE_CSV = OUTPUT_DIR / "year_coverage.csv"

print(f"Input: {RAW_CSV.name} ({RAW_CSV.stat().st_size / 1024**3:.2f} GB)")
print(f"Output: {CLEAN_PARQUET.relative_to(PROJECT_DIR)}")

Input: acled_global_political_violence_2020-01-01_2026-07-31_download-2026-08-25.csv.csv (1.03 GB)
Output: data/processed/acled_civilian_targeting_2021_2025.parquet


## 2. Selezione delle variabili

Escludiamo `notes`, `tags` e il testo completo delle fonti dal dataset analitico: sono campi pesanti e non necessari alle research questions definite. Manteniamo invece identificativi, tempo, geografia, tipologia dell'evento, attori, codici di interazione, precisione e fatalità. Il CSV originale resta invariato.

In [4]:
USECOLS = [
    "event_id_cnty", "event_date", "year", "time_precision",
    "disorder_type", "event_type", "sub_event_type",
    "actor1", "assoc_actor_1", "inter1",
    "actor2", "assoc_actor_2", "inter2", "interaction",
    "civilian_targeting", "iso", "region", "country",
    "admin1", "admin2", "admin3", "location",
    "latitude", "longitude", "geo_precision",
    "source_scale", "fatalities", "timestamp",
]

preview = pd.read_csv(RAW_CSV, nrows=5, usecols=USECOLS, dtype="string")
display(preview)
print(f"Variabili selezionate: {len(USECOLS)}")

,event_id_cnty,event_date,year,time_precision,disorder_type,event_type,sub_event_type,actor1,assoc_actor_1,inter1,actor2,assoc_actor_2,inter2,interaction,civilian_targeting,iso,region,country,admin1,admin2,admin3,location,latitude,longitude,geo_precision,source_scale,fatalities,timestamp
0,TUR10582,2020-01-01,2020,2,Strategic developments,Strategic developments,Arrests,Military Forces of Turkey (2016-) Gendarmerie,Military Forces of Turkey (2016-); Police Forces of Turkey (2016-),1,Civilians (International),Refugees/IDPs (International),7,17,<NA>,792,Middle East,Turkey,Edirne,Edirne,<NA>,Edirne,41.6757,26.5587,3,National,0,1578503874
1,TUR10583,2020-01-01,2020,2,Strategic developments,Strategic developments,Arrests,Military Forces of Turkey (2016-),<NA>,1,Civilians (International),Refugees/IDPs (International),7,17,<NA>,792,Middle East,Turkey,Aydin,Kusadasi,<NA>,Kusadasi,37.8582,27.2607,3,National,0,1578503874
2,IRN5946,2020-01-01,2020,1,Demonstrations,Protests,Peaceful protest,Protesters (Iran),Students (Iran),6,<NA>,<NA>,0,60,<NA>,364,Middle East,Iran,Fars,Shiraz,Central,Shiraz,29.6103,52.5311,1,New media,0,1578503875
3,IRN5874,2020-01-01,2020,1,Demonstrations,Protests,Peaceful protest,Protesters (Iran),Health Workers (Iran),6,<NA>,<NA>,0,60,<NA>,364,Middle East,Iran,Razavi Khorasan,Mashhad,Central,Mashhad,36.3156,59.5680,1,National,0,1578503875
4,TUN6016,2020-01-01,2020,1,Demonstrations,Protests,Peaceful protest,Protesters (Tunisia),UGTT: Tunisian General Labour Union; Labor Group (Tunisia),6,<NA>,<NA>,0,60,<NA>,788,Northern Africa,Tunisia,Tataouine,Tataouine Nord,<NA>,Tataouine,32.9297,10.4518,2,National,0,1578512391


Variabili selezionate: 28


## 3. Caricamento efficiente e filtro

Il file supera 1 GB. Lo leggiamo a blocchi e filtriamo immediatamente periodo e tag, così in memoria entrano soltanto le osservazioni rilevanti. Il filtro è esatto (`Civilian targeting`) per evitare di interpretare come positivi valori inattesi.

In [5]:
filtered_chunks = []
load_audit_rows = []

reader = pd.read_csv(
    RAW_CSV,
    usecols=USECOLS,
    dtype="string",
    chunksize=CHUNK_SIZE,
    encoding="utf-8-sig",
)

for chunk_number, chunk in enumerate(reader, start=1):
    raw_year = pd.to_numeric(chunk["year"], errors="coerce")
    in_period = raw_year.between(START_YEAR, END_YEAR, inclusive="both")
    targeting_value = chunk["civilian_targeting"].str.strip()
    is_civilian_targeting = targeting_value.eq("Civilian targeting")
    keep = (
        in_period.fillna(False).astype(bool)
        & is_civilian_targeting.fillna(False).astype(bool)
    )

    filtered_chunks.append(chunk.loc[keep].copy())
    load_audit_rows.append({
        "chunk": chunk_number,
        "rows_read": len(chunk),
        "rows_in_period": int(in_period.sum()),
        "rows_kept": int(keep.sum()),
    })

df = pd.concat(filtered_chunks, ignore_index=True)
load_audit = pd.DataFrame(load_audit_rows)

display(load_audit.tail())
display(load_audit[["rows_read", "rows_in_period", "rows_kept"]].sum().to_frame("total"))
print(f"Righe filtrate caricate: {len(df):,}")

,chunk,rows_read,rows_in_period,rows_kept
5,6,200000,200000,31434
6,7,200000,200000,29442
7,8,200000,200000,31180
8,9,200000,200000,31316
9,10,114454,114454,18984


,total
rows_read,1914454
rows_in_period,1644002
rows_kept,251579


Righe filtrate caricate: 251,579


## 4. Normalizzazione dei tipi e dei valori

In [6]:
# Rimuove spazi esterni e converte stringhe vuote in valori mancanti.
text_columns = [
    "event_id_cnty", "disorder_type", "event_type", "sub_event_type",
    "actor1", "assoc_actor_1", "actor2", "assoc_actor_2",
    "civilian_targeting", "region", "country", "admin1",
    "admin2", "admin3", "location", "source_scale",
]
for column in text_columns:
    df[column] = df[column].str.strip().replace("", pd.NA)

# Data: event_date è la fonte canonica per anno, mese e trimestre.
df["event_date"] = pd.to_datetime(df["event_date"], errors="coerce")
df["year_reported"] = pd.to_numeric(df["year"], errors="coerce").astype("Int16")
df["year"] = df["event_date"].dt.year.astype("Int16")
df["month"] = df["event_date"].dt.month.astype("Int8")
df["quarter"] = df["event_date"].dt.quarter.astype("Int8")
df["year_month"] = df["event_date"].dt.to_period("M").astype("string")
df["year_mismatch"] = df["year"].ne(df["year_reported"]).fillna(True)

# Numeri: i valori non validi restano NA; non assumiamo che un dato mancante equivalga a zero.
df["fatalities"] = pd.to_numeric(df["fatalities"], errors="coerce").astype("Int64")
df["fatalities_invalid"] = df["fatalities"].lt(0).fillna(False)
df.loc[df["fatalities_invalid"], "fatalities"] = pd.NA

for column in ["latitude", "longitude"]:
    df[column] = pd.to_numeric(df[column], errors="coerce")
df["coordinates_valid"] = (
    df["latitude"].between(-90, 90)
    & df["longitude"].between(-180, 180)
).fillna(False)
df.loc[~df["coordinates_valid"], ["latitude", "longitude"]] = np.nan

integer_columns = ["time_precision", "inter1", "inter2", "interaction", "geo_precision", "timestamp"]
for column in integer_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce").astype("Int64")

df.info(memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 251579 entries, 0 to 251578
Data columns (total 35 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   event_id_cnty       251579 non-null  string        
 1   event_date          251579 non-null  datetime64[ns]
 2   year                251579 non-null  Int16         
 3   time_precision      251579 non-null  Int64         
 4   disorder_type       251579 non-null  string        
 5   event_type          251579 non-null  string        
 6   sub_event_type      251579 non-null  string        
 7   actor1              251579 non-null  string        
 8   assoc_actor_1       37090 non-null   string        
 9   inter1              251579 non-null  Int64         
 10  actor2              251579 non-null  string        
 11  assoc_actor_2       104236 non-null  string        
 12  inter2              251579 non-null  Int64         
 13  interaction         251579 no

## 5. Duplicati e righe non analizzabili

L'identificativo ACLED `event_id_cnty` deve essere unico. Rimuoviamo eventuali duplicati mantenendo il record con timestamp più recente; scartiamo soltanto righe prive di ID/data, fuori periodo secondo la data effettiva o con anno incoerente. Le coordinate mancanti non causano l'eliminazione dell'evento: lo rendono solo non utilizzabile nelle mappe puntuali.

In [7]:
rows_before_rules = len(df)
duplicate_ids = int(df.duplicated(subset="event_id_cnty", keep=False).sum())

df = (
    df.sort_values(["event_id_cnty", "timestamp"], na_position="first")
      .drop_duplicates(subset="event_id_cnty", keep="last")
)

valid_core = (
    df["event_id_cnty"].notna()
    & df["event_date"].notna()
    & df["year"].between(START_YEAR, END_YEAR, inclusive="both")
    & ~df["year_mismatch"]
)
rows_invalid_core = int((~valid_core).sum())
df = df.loc[valid_core].copy()

print(f"Righe prima delle regole: {rows_before_rules:,}")
print(f"Righe coinvolte in ID duplicati: {duplicate_ids:,}")
print(f"Righe rimosse per campi core non validi: {rows_invalid_core:,}")
print(f"Righe pulite: {len(df):,}")

Righe prima delle regole: 251,579
Righe coinvolte in ID duplicati: 0
Righe rimosse per campi core non validi: 0
Righe pulite: 251,579


## 6. Feature engineering per le research questions

In [8]:
# Categorie ACLED dei tipi di attore (codici inter1/inter2).
ACTOR_TYPE_MAP = {
    1: "State forces",
    2: "Rebel groups",
    3: "Political militias",
    4: "Identity militias",
    5: "Rioters",
    6: "Protesters",
    7: "Civilians",
    8: "External/other forces",
}
df["actor1_type"] = df["inter1"].map(ACTOR_TYPE_MAP).astype("string")
df["actor2_type"] = df["inter2"].map(ACTOR_TYPE_MAP).astype("string")

# ISO numerico ACLED -> ISO alpha-3, utile per choropleth Plotly.
def numeric_iso_to_alpha3(value):
    if pd.isna(value):
        return pd.NA
    numeric = str(value).split(".")[0].zfill(3)
    match = pycountry.countries.get(numeric=numeric)
    return match.alpha_3 if match else pd.NA

df["iso3"] = df["iso"].map(numeric_iso_to_alpha3).astype("string")

# Indicatori di impatto: descrittivi, non una classificazione normativa del rischio.
df["civilian_targeting_flag"] = True
df["event_had_fatalities"] = df["fatalities"].gt(0).fillna(False)
df["fatality_band"] = pd.cut(
    df["fatalities"],
    bins=[-0.1, 0, 1, 4, 9, np.inf],
    labels=["0", "1", "2–4", "5–9", "10+"],
)

# Finestra comune per confrontare il 2025 parziale con gli anni precedenti.
latest_date = df["event_date"].max()
last_year_rows = df["year"].eq(END_YEAR)
last_year_max_date = df.loc[last_year_rows, "event_date"].max()
cutoff_month = int(last_year_max_date.month)
cutoff_day = int(last_year_max_date.day)
df["within_common_annual_window"] = (
    df["month"].lt(cutoff_month)
    | (df["month"].eq(cutoff_month) & df["event_date"].dt.day.le(cutoff_day))
)

print(f"Ultima data disponibile: {latest_date.date()}")
print(f"Finestra comparabile: 1 gennaio–{cutoff_day:02d}/{cutoff_month:02d} di ogni anno")

Ultima data disponibile: 2025-08-25
Finestra comparabile: 1 gennaio–25/08 di ogni anno


## 7. Coverage e quality audit

In [9]:
year_coverage = (
    df.groupby("year", observed=True)
      .agg(
          first_date=("event_date", "min"),
          last_date=("event_date", "max"),
          events=("event_id_cnty", "size"),
          fatalities=("fatalities", "sum"),
      )
      .reset_index()
)
year_coverage["is_partial_year"] = year_coverage["last_date"].dt.strftime("%m-%d").ne("12-31")
display(year_coverage)

quality_audit = pd.DataFrame({
    "check": [
        "rows_clean", "unique_event_ids", "duplicate_event_ids",
        "missing_event_date", "year_mismatch", "missing_fatalities",
        "negative_fatalities_found", "invalid_or_missing_coordinates",
        "missing_iso3", "first_event_date", "last_event_date",
    ],
    "value": [
        len(df), df["event_id_cnty"].nunique(), duplicate_ids,
        int(df["event_date"].isna().sum()), int(df["year_mismatch"].sum()),
        int(df["fatalities"].isna().sum()), int(df["fatalities_invalid"].sum()),
        int((~df["coordinates_valid"]).sum()), int(df["iso3"].isna().sum()),
        df["event_date"].min().date().isoformat(),
        df["event_date"].max().date().isoformat(),
    ],
})
display(quality_audit)

,year,first_date,last_date,events,fatalities,is_partial_year
0,2021,2021-01-01,2021-12-31,41510,44645,False
1,2022,2022-01-01,2022-12-31,51356,55704,False
2,2023,2023-01-01,2023-12-31,54735,71692,False
3,2024,2024-01-01,2024-12-31,60181,78960,False
4,2025,2025-01-01,2025-08-25,43797,52967,True


,check,value
0,rows_clean,251579
1,unique_event_ids,251579
2,duplicate_event_ids,0
3,missing_event_date,0
4,year_mismatch,0
5,missing_fatalities,0
6,negative_fatalities_found,0
7,invalid_or_missing_coordinates,0
8,missing_iso3,167
9,first_event_date,2021-01-01


In [10]:
# Test automatici: il notebook si interrompe se le assunzioni principali non sono vere.
assert df["event_id_cnty"].is_unique, "event_id_cnty non è univoco"
assert df["event_date"].notna().all(), "Sono presenti date mancanti/non valide"
assert df["year"].between(START_YEAR, END_YEAR).all(), "Sono presenti anni fuori periodo"
assert df["civilian_targeting"].eq("Civilian targeting").all(), "Filtro civilian targeting non rispettato"
assert df["fatalities"].dropna().ge(0).all(), "Sono presenti fatalità negative"
assert df.loc[df["coordinates_valid"], "latitude"].between(-90, 90).all()
assert df.loc[df["coordinates_valid"], "longitude"].between(-180, 180).all()

print("✓ Tutti i controlli di integrità sono superati.")

✓ Tutti i controlli di integrità sono superati.


## 8. Ottimizzazione ed esportazione

In [11]:
# Categorie ripetitive: riducono la memoria e accelerano groupby/visualizzazioni.
category_columns = [
    "disorder_type", "event_type", "sub_event_type", "civilian_targeting",
    "region", "country", "source_scale", "actor1_type", "actor2_type",
    "fatality_band",
]
for column in category_columns:
    df[column] = df[column].astype("category")

df = df.sort_values(["event_date", "event_id_cnty"]).reset_index(drop=True)
df.to_parquet(CLEAN_PARQUET, index=False, compression="zstd")
quality_audit.to_csv(AUDIT_CSV, index=False)
year_coverage.to_csv(COVERAGE_CSV, index=False)

print(f"Dataset pulito: {CLEAN_PARQUET} ({CLEAN_PARQUET.stat().st_size / 1024**2:.1f} MB)")
print(f"Audit: {AUDIT_CSV}")
print(f"Coverage: {COVERAGE_CSV}")

Dataset pulito: /Users/nicole/Desktop/projects/datavis/data/processed/acled_civilian_targeting_2021_2025.parquet (6.9 MB)
Audit: /Users/nicole/Desktop/projects/datavis/data/processed/cleaning_audit.csv
Coverage: /Users/nicole/Desktop/projects/datavis/data/processed/year_coverage.csv


## 9. Dataset pronto per l'analisi

La tabella seguente è soltanto un controllo finale, non l'analisi conclusiva. Mostra le metriche che alimenteranno timeline, ranking, scatter frequenza–gravità e matrice di priorità umanitaria.

In [12]:
country_preview = (
    df.groupby(["region", "country"], observed=True)
      .agg(
          events=("event_id_cnty", "size"),
          total_fatalities=("fatalities", "sum"),
          mean_fatalities_per_event=("fatalities", "mean"),
          fatal_events=("event_had_fatalities", "sum"),
      )
      .sort_values(["total_fatalities", "events"], ascending=False)
      .head(15)
)
display(country_preview)
display(df.head())

,,events,total_fatalities,mean_fatalities_per_event,fatal_events
region,country,,,,
Middle East,Palestine,19903,50602,2.542431,7161
North America,Mexico,27553,30489,1.106558,21653
South America,Brazil,23830,20893,0.876752,18424
Western Africa,Nigeria,11166,18734,1.677772,5801
Middle Africa,Democratic Republic of Congo,7908,17741,2.243424,5195
Southeast Asia,Myanmar,16282,15830,0.972239,8120
Northern Africa,Sudan,6295,15702,2.494361,3684
Europe,Ukraine,15310,11826,0.772436,5363
Eastern Africa,Ethiopia,2690,10950,4.070632,1945


,event_id_cnty,event_date,year,time_precision,disorder_type,event_type,sub_event_type,actor1,assoc_actor_1,inter1,actor2,assoc_actor_2,inter2,interaction,civilian_targeting,iso,region,country,admin1,admin2,admin3,location,latitude,longitude,geo_precision,source_scale,fatalities,timestamp,year_reported,month,quarter,year_month,year_mismatch,fatalities_invalid,coordinates_valid,actor1_type,actor2_type,iso3,civilian_targeting_flag,event_had_fatalities,fatality_band,within_common_annual_window
0,AFG50076,2021-01-01,2021,1,Political violence,Violence against civilians,Attack,Unidentified Armed Group (Afghanistan),<NA>,3,Civilians (Afghanistan),Journalists (Afghanistan),7,37,Civilian targeting,4,Caucasus and Central Asia,Afghanistan,Ghor,Chighcheran,<NA>,Chighcheran,34.5195,65.2509,1,National-International,1,1754409046,2021,1,1,2021-01,False,False,True,Political militias,Civilians,AFG,True,True,1,True
1,AFG50185,2021-01-01,2021,1,Political violence,Violence against civilians,Attack,Unidentified Armed Group (Afghanistan),<NA>,3,Civilians (Afghanistan),Police Forces of Afghanistan (2014-2021),7,37,Civilian targeting,4,Caucasus and Central Asia,Afghanistan,Kandahar,Kandahar,<NA>,Kandahar,31.6133,65.7101,1,National,1,1754409046,2021,1,1,2021-01,False,False,True,Political militias,Civilians,AFG,True,True,1,True
2,AFG59611,2021-01-01,2021,1,Political violence,Violence against civilians,Attack,Unidentified Armed Group (Afghanistan),<NA>,3,Civilians (Afghanistan),<NA>,7,37,Civilian targeting,4,Caucasus and Central Asia,Afghanistan,Jowzjan,Aqchah,<NA>,Aqchah,36.905,66.1834,2,Local partner-Other,1,1754409046,2021,1,1,2021-01,False,False,True,Political militias,Civilians,AFG,True,True,1,True
3,AZE17124,2021-01-01,2021,1,Political violence,Explosions/Remote violence,Remote explosive/landmine/IED,Unidentified Military Forces,<NA>,8,Civilians (Azerbaijan),<NA>,7,78,Civilian targeting,31,Caucasus and Central Asia,Azerbaijan,Fizuli,<NA>,<NA>,Fizuli,39.5982,47.1469,2,National,0,1754409061,2021,1,1,2021-01,False,False,True,External/other forces,Civilians,AZE,True,False,0,True
4,BGD18666,2021-01-01,2021,1,Political violence,Violence against civilians,Attack,Unidentified Armed Group (Bangladesh),<NA>,3,Civilians (Bangladesh),BCL: Bangladesh Chhatra League; Students (Bangladesh),7,37,Civilian targeting,50,South Asia,Bangladesh,Chittagong,Cox's Bazar,Teknaf,Teknaf,20.8583,92.2977,2,National,1,1754409061,2021,1,1,2021-01,False,False,True,Political militias,Civilians,BGD,True,True,1,True
